# Exercises XP Gold — Advanced Prompt Engineering

## Solution académique complète

Ce notebook complète les six exercices autour de :

- la correction de raisonnements ;
- le choix de patterns de prompting ;
- la comparaison de plusieurs chemins de résolution ;
- le prompt chaining avec logique conditionnelle ;
- la réduction des biais ;
- la simulation d’une mémoire conversationnelle.

Le code est commenté afin d’expliquer le rôle de chaque fonction et de
chaque validation.

## Objectifs pédagogiques

À la fin du notebook, vous saurez :

1. repérer une erreur dans une suite de calculs ;
2. demander une justification courte et vérifiable sans exiger un
   raisonnement interne illimité ;
3. choisir un pattern de prompt adapté à une classification ;
4. comparer plusieurs méthodes indépendantes ;
5. construire un workflow à plusieurs étapes ;
6. ajouter des branches conditionnelles ;
7. réduire les stéréotypes dans des recommandations ;
8. structurer une mémoire conversationnelle réutilisable.

## Note méthodologique sur le Chain-of-Thought

Dans ce notebook, les prompts demandent des **étapes de calcul courtes,
vérifiables et orientées vers le résultat**.

L’objectif n’est pas d’obtenir un raisonnement interne exhaustif, mais de
produire :

- les données utilisées ;
- les opérations principales ;
- une vérification indépendante ;
- la réponse finale.

Cette approche est plus facile à auditer et réduit les explications longues
ou incohérentes.

# Helper — Exécuter ou afficher un prompt

Le notebook peut utiliser Ollama si l’outil est installé localement.

Dans Google Colab, Ollama n’est généralement pas disponible. La fonction
passe donc automatiquement en **dry-run** et affiche le prompt sans faire
échouer le notebook.

In [ ]:
import json
import os
import re
import subprocess
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional


def run_prompt(
    prompt: str,
    model: Optional[str] = None,
    temperature: float = 0.2,
    max_new_tokens: int = 200,
) -> Optional[str]:
    """Execute a prompt with Ollama when available.

    Parameters
    ----------
    prompt:
        Complete text sent to the model.
    model:
        Ollama model name. If omitted, the OLLAMA_MODEL environment
        variable is used, with 'llama3' as fallback.
    temperature:
        Kept in the function signature for documentation. The basic
        `ollama run` CLI call used here does not directly apply it.
    max_new_tokens:
        Also documented for consistency with common LLM APIs.

    Returns
    -------
    Optional[str]
        Model output when Ollama succeeds; otherwise None.
    """
    # Validate the input early to avoid sending an empty prompt.
    if not prompt or not prompt.strip():
        raise ValueError("The prompt cannot be empty.")

    selected_model = model or os.environ.get("OLLAMA_MODEL", "llama3")
    command = ["ollama", "run", selected_model]

    try:
        # Send the prompt through standard input and capture both outputs.
        process = subprocess.run(
            command,
            input=prompt.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )

        output = process.stdout.decode(
            "utf-8",
            errors="ignore",
        ).strip()

        print(output)
        return output

    except FileNotFoundError:
        # Colab normally reaches this branch because Ollama is not installed.
        print("[dry-run] Ollama is not installed.")
        print("[dry-run] Prompt that would be sent:\n")
        print(prompt)

    except subprocess.CalledProcessError as error:
        # A model download or local Ollama error should not stop the notebook.
        error_message = (
            error.stderr.decode("utf-8", errors="ignore")
            if error.stderr
            else str(error)
        )
        print("[dry-run] Ollama call failed:", error_message)
        print("\nPrompt that would be sent:\n")
        print(prompt)

    return None

# Exercise 1 — Debug a Faulty Chain-of-Thought

## Problem

- One pencil costs **$0.75**.
- Alice buys **6 pencils**.
- She pays **$5.00**.

Incorrect reasoning supplied:

```text
6 × $0.75 = $4.75
$5.00 - $4.75 = $0.50
```

## 1.1 Identify the mistakes

In [ ]:
cot_issue = {
    "multiplication_error": (
        "6 × $0.75 equals $4.50, not $4.75."
    ),
    "subtraction_error": (
        "$5.00 - $4.75 would equal $0.25, not $0.50."
    ),
    "main_cause": (
        "The first arithmetic error changes the purchase total, and the "
        "second line is also internally inconsistent."
    ),
}

# Display each issue clearly for the written submission.
for issue_name, explanation in cot_issue.items():
    print(f"{issue_name}: {explanation}")

## 1.2 Corrected prompt

Le prompt demande uniquement les opérations nécessaires et une vérification
rapide.

In [ ]:
fixed_cot_prompt = """
Solve the following money problem using short, verifiable calculation steps.

Problem:
A shop sells pencils at $0.75 each. Alice buys 6 pencils and pays with a
$5.00 bill. How much change should she receive?

Required method:
1. Calculate the total price as quantity × unit price.
2. Calculate the change as amount paid − total price.
3. Verify that total price + change = amount paid.
4. Return the final answer in dollars with two decimal places.

Do not skip any arithmetic operation.
""".strip()

print(fixed_cot_prompt)

## 1.3 Deterministic verification in Python

In [ ]:
# Store money as integer cents to avoid floating-point rounding problems.
unit_price_cents = 75
quantity = 6
amount_paid_cents = 500

# Step 1: Calculate the purchase total.
total_cost_cents = unit_price_cents * quantity

# Step 2: Calculate the change.
correct_change_cents = amount_paid_cents - total_cost_cents

# Step 3: Verify the accounting identity.
assert total_cost_cents + correct_change_cents == amount_paid_cents

correct_change = f"${correct_change_cents / 100:.2f}"

print("Total cost:", f"${total_cost_cents / 100:.2f}")
print("Correct change:", correct_change)

## 1.4 Correct answer

\[
6 \times 0.75 = 4.50
\]

\[
5.00 - 4.50 = 0.50
\]

**Alice receives $0.50.**

# Exercise 2 — Choose the Right Prompt Pattern

## Use case

Classify a customer message into exactly one of:

- Billing Issue
- Technical Support
- Account Access
- Other

## 2.1 Selected pattern: Few-Shot Classification

Few-shot prompting is appropriate because:

- class boundaries can be ambiguous ;
- examples make the desired distinctions concrete ;
- a strict output schema improves consistency ;
- unseen messages can still be generalized from the definitions and
  demonstrations.

A zero-shot prompt could work, but would be more sensitive to ambiguous
wording such as “I cannot use the service because my payment failed.”

In [ ]:
chosen_pattern = "Few-Shot Classification with label definitions"

classification_prompt_template = """
Act as a customer-support ticket classifier.

Assign the customer message to exactly one label:

1. Billing Issue
   Payments, charges, invoices, refunds, subscriptions, prices, or failed
   transactions.

2. Technical Support
   Product errors, bugs, crashes, performance problems, or features not
   working after the user has access.

3. Account Access
   Login problems, passwords, verification codes, locked accounts, or
   authentication.

4. Other
   Messages that do not primarily match the three categories above.

Decision rule:
- Choose the category representing the main problem.
- If a login or authentication failure prevents access, choose
  "Account Access".
- If the main issue is a payment or charge, choose "Billing Issue".
- Do not create new labels.

Examples:

Customer: "I was charged twice for the same monthly plan."
Output: {"label": "Billing Issue"}

Customer: "The app closes every time I upload a photo."
Output: {"label": "Technical Support"}

Customer: "My verification code never arrives, so I cannot sign in."
Output: {"label": "Account Access"}

Customer: "Do you offer discounts for nonprofit organizations?"
Output: {"label": "Other"}

Customer message:
<message>
{customer_message}
</message>

Return valid JSON only:
{"label": "<one allowed label>"}
""".strip()

justification = (
    "Few-shot examples reduce ambiguity and improve label consistency. "
    "Definitions support generalization, while the JSON-only schema makes "
    "the output easy to validate and integrate into software."
)

print("Pattern:", chosen_pattern)
print("\nJustification:", justification)

## 2.2 Build and inspect a test prompt

In [ ]:
sample_customer_message = (
    "I reset my password, but the new password is still rejected."
)

# Insert the real customer message into the reusable template.
classification_prompt = classification_prompt_template.format(
    customer_message=sample_customer_message
)

print(classification_prompt)

## 2.3 Output validator

Le validateur vérifie que la réponse est un JSON et que le label appartient
à la liste autorisée.

In [ ]:
ALLOWED_LABELS = {
    "Billing Issue",
    "Technical Support",
    "Account Access",
    "Other",
}


def validate_classification_output(output: str) -> Dict[str, Any]:
    """Validate a JSON classification response.

    The function raises a clear error when:
    - the text is not valid JSON;
    - the `label` key is missing;
    - the label is outside the allowed taxonomy.
    """
    try:
        parsed_output = json.loads(output)
    except json.JSONDecodeError as error:
        raise ValueError("The model output is not valid JSON.") from error

    if set(parsed_output.keys()) != {"label"}:
        raise ValueError(
            "The output must contain exactly one key: 'label'."
        )

    if parsed_output["label"] not in ALLOWED_LABELS:
        raise ValueError(
            f"Unsupported label: {parsed_output['label']}"
        )

    return parsed_output


# Example of a valid model response.
simulated_classification_output = '{"label": "Account Access"}'

validated_classification = validate_classification_output(
    simulated_classification_output
)

print(validated_classification)

# Exercise 3 — AlignedCoT with Multiple Reasoning Paths

## Problem

- 2 small pots at $2 each ;
- 3 medium pots at $4 each ;
- 1 large pot at $6 each.

## 3.1 Aligned multi-path prompt

Les deux chemins utilisent des structures différentes :

- chemin A : sous-totaux par catégorie ;
- chemin B : développement direct d’une expression unique.

Une troisième étape compare les résultats.

In [ ]:
aligned_cot_prompt = """
Solve the flower-pot cost problem using two independent, concise methods.

Data:
- Small pots: 2 pots at $2 each
- Medium pots: 3 pots at $4 each
- Large pots: 1 pot at $6 each

Path A — Category subtotals:
1. Calculate the subtotal for each pot size.
2. Add the three subtotals.

Path B — Single expression:
1. Write one complete arithmetic expression using all quantities and prices.
2. Evaluate the expression.

Alignment check:
- Compare the answers from Path A and Path B.
- If they match, report the common result.
- If they differ, identify the arithmetic error and recompute.
- Return the final answer in dollars.

Keep each path brief and show only the necessary calculations.
""".strip()

print(aligned_cot_prompt)

## 3.2 Independent calculations

In [ ]:
# Path A: compute each category separately.
small_subtotal = 2 * 2
medium_subtotal = 3 * 4
large_subtotal = 1 * 6
path_a_total = (
    small_subtotal
    + medium_subtotal
    + large_subtotal
)

# Path B: evaluate one combined expression.
path_b_total = (2 * 2) + (3 * 4) + (1 * 6)

# Alignment check: both independent structures must agree.
paths_are_aligned = path_a_total == path_b_total

assert paths_are_aligned, "The two calculation paths disagree."

aligned_answer = f"${path_a_total:.2f}"

print("Path A total:", path_a_total)
print("Path B total:", path_b_total)
print("Aligned:", paths_are_aligned)
print("Final answer:", aligned_answer)

## 3.3 Correct answer

- Small pots: \(2 \times 2 = 4\)
- Medium pots: \(3 \times 4 = 12\)
- Large pot: \(1 \times 6 = 6\)

\[
4 + 12 + 6 = 22
\]

**The total cost is $22.00.**

# Exercise 4 — Multi-Step Academic Document Pipeline

## Required stages

1. Identify the domain.
2. Extract the main contributions.
3. Generate a follow-up research question.

## 4.1 Stage 1 — Domain classification

In [ ]:
stage1_domain = """
Act as an academic paper triage assistant.

Read the paper title and abstract below.

<title>
{title}
</title>

<abstract>
{abstract}
</abstract>

Choose exactly one primary domain:
- Biology
- Physics
- Computer Science
- Chemistry
- Medicine
- Social Science
- Interdisciplinary
- Other

Also provide:
- confidence: a number from 0.00 to 1.00;
- evidence: up to 3 short phrases from the title or abstract;
- secondary_domain: one label or null.

Return valid JSON only:
{
  "primary_domain": "...",
  "secondary_domain": "... or null",
  "confidence": 0.00,
  "evidence": ["...", "..."]
}
""".strip()

print(stage1_domain)

## 4.2 Stage 2 — Contribution extraction

In [ ]:
stage2_contrib = """
Act as a research-methodology reviewer specializing in {primary_domain}.

Paper title:
{title}

Abstract:
{abstract}

Domain classification from Stage 1:
{domain_result}

Extract only contributions explicitly supported by the abstract.

Return:
1. Research problem — one sentence.
2. Method or approach — one sentence.
3. Main contributions — 1 to 3 bullets.
4. Evidence for each contribution — a short phrase from the abstract.
5. Reported limitations — use "Not stated in the abstract" when absent.

Do not:
- invent experimental results;
- infer performance numbers;
- claim novelty unless the abstract claims it;
- use external knowledge.
""".strip()

print(stage2_contrib)

## 4.3 Stage 3 — Follow-up research question

In [ ]:
stage3_followup = """
Act as a research agenda designer in {primary_domain}.

Use the following information:

Title:
{title}

Abstract:
{abstract}

Extracted contributions:
{contribution_result}

Generate exactly one follow-up research question that:
- directly extends or tests one stated contribution;
- is not already answered by the abstract;
- names the population, system, condition, or variable to investigate;
- is feasible enough to guide a future study;
- does not assume facts absent from the abstract.

Output format:
Research question: ...
Rationale: one sentence.
Suggested evidence: one sentence describing the data or experiment needed.
""".strip()

print(stage3_followup)

## 4.4 Conditional logic and context chaining

La sortie de chaque étape devient le contexte de la suivante.

Branches utiles :

- confiance de domaine faible → demander une revue humaine ;
- domaine interdisciplinaire → injecter les deux domaines dans l’étape 2 ;
- abstract trop court → arrêter l’extraction détaillée ;
- aucune limitation indiquée → créer une question à partir d’une lacune
  explicitement observable, sans inventer de limitation.

In [ ]:
def choose_pipeline_action(
    domain_result: Dict[str, Any],
    abstract: str,
) -> str:
    """Choose the next workflow action using simple business rules."""
    confidence = float(domain_result.get("confidence", 0.0))
    primary_domain = domain_result.get("primary_domain")
    abstract_word_count = len(abstract.split())

    # Very short abstracts do not support reliable contribution extraction.
    if abstract_word_count < 40:
        return "request_more_document_text"

    # Low-confidence classification should not silently continue.
    if confidence < 0.60:
        return "human_domain_review"

    # Interdisciplinary work needs a broader reviewer configuration.
    if primary_domain == "Interdisciplinary":
        return "run_interdisciplinary_extraction"

    return "run_standard_extraction"


# Simulated output from Stage 1.
sample_domain_result = {
    "primary_domain": "Computer Science",
    "secondary_domain": None,
    "confidence": 0.91,
    "evidence": [
        "neural retrieval model",
        "benchmark datasets",
    ],
}

sample_abstract = (
    "This study introduces a neural retrieval model for academic search. "
    "The approach combines dense representations with metadata filters and "
    "is evaluated on multiple benchmark datasets. Results show improved "
    "retrieval accuracy compared with the reported baselines. The abstract "
    "also describes an ablation study and discusses computational cost."
)

next_action = choose_pipeline_action(
    sample_domain_result,
    sample_abstract,
)

print("Pipeline action:", next_action)

## 4.5 Reusable context object

In [ ]:
@dataclass
class PaperPipelineContext:
    """State passed from one prompt stage to the next."""

    title: str
    abstract: str
    domain_result: Optional[Dict[str, Any]] = None
    contribution_result: Optional[Dict[str, Any]] = None
    follow_up_result: Optional[Dict[str, Any]] = None


# The context object avoids losing or mixing information between stages.
paper_context = PaperPipelineContext(
    title="Hybrid Retrieval for Academic Search",
    abstract=sample_abstract,
    domain_result=sample_domain_result,
)

print(json.dumps(asdict(paper_context), indent=2))

# Exercise 5 — Role Prompting to Reduce Bias

## Career recommendation scenario

Le système doit recommander des carrières à partir des compétences et des
intérêts, sans utiliser de stéréotypes liés au genre ou à d’autres
caractéristiques protégées.

## 5.1 Basic prompt that may amplify bias

In [ ]:
biased_prompt = """
Look at this user's profile and recommend the three careers that seem most
suitable.

User profile:
{user_profile}

Explain your choices.
""".strip()

print(biased_prompt)

Ce prompt n’est pas explicitement discriminatoire, mais il laisse le modèle
utiliser des associations apprises et ne définit aucun critère de décision.

## 5.2 Revised role-based prompt

In [ ]:
debiased_prompt = """
Act as an evidence-based career guidance counselor and fairness reviewer.

Recommend career paths using only the user's stated:
- skills;
- interests;
- education or experience;
- work preferences;
- accessibility or schedule needs voluntarily provided by the user.

User profile:
<profile>
{user_profile}
</profile>

Fairness rules:
1. Do not use or infer gender, race, ethnicity, age, religion, disability,
   family status, or socioeconomic background as evidence of ability.
2. Do not rely on occupational stereotypes.
3. Apply the same skill-to-career criteria to every user.
4. Include at least one non-obvious option when it is supported by the
   profile.
5. If important information is missing, state the gap instead of assuming.
6. Do not rank a career lower because a demographic group is
   underrepresented in that field.

Output exactly 4 recommendations in a table with:
- career;
- matching evidence from the profile;
- possible skill gap;
- one practical next step.

After the table, add a short fairness check confirming which profile
evidence was used and that protected characteristics were not used.
""".strip()

fairness_explanation = (
    "The role combines career guidance with an explicit fairness review. "
    "It limits the evidence to skills and preferences, prohibits protected "
    "attributes and stereotypes, requires transparent matching evidence, "
    "and adds a final audit of the recommendation criteria."
)

print(debiased_prompt)
print("\nExplanation:", fairness_explanation)

## 5.3 Simple prompt audit

In [ ]:
PROTECTED_ATTRIBUTE_TERMS = {
    "gender",
    "race",
    "ethnicity",
    "religion",
    "age",
    "disability",
    "family status",
}


def audit_fairness_prompt(prompt: str) -> Dict[str, Any]:
    """Check whether core fairness safeguards appear in a prompt.

    This is a basic keyword-based audit, not a complete fairness evaluation.
    """
    normalized_prompt = prompt.lower()

    protected_terms_mentioned = sorted(
        term
        for term in PROTECTED_ATTRIBUTE_TERMS
        if term in normalized_prompt
    )

    return {
        "mentions_protected_attributes": protected_terms_mentioned,
        "prohibits_stereotypes": (
            "stereotype" in normalized_prompt
        ),
        "requires_evidence": (
            "evidence" in normalized_prompt
        ),
        "requires_fairness_check": (
            "fairness check" in normalized_prompt
        ),
        "handles_missing_information": (
            "missing" in normalized_prompt
        ),
    }


fairness_audit = audit_fairness_prompt(debiased_prompt)
print(json.dumps(fairness_audit, indent=2))

## 5.4 Why the role improves fairness

Le rôle révisé :

- remplace l’intuition générale par des critères fondés sur les preuves ;
- interdit l’usage de caractéristiques protégées ;
- exige une justification liée au profil ;
- demande une option moins évidente lorsqu’elle est pertinente ;
- ajoute un contrôle de cohérence à la fin.

Il réduit le risque de biais, sans garantir à lui seul une absence totale de
discrimination. Des tests sur des profils contrefactuels restent
nécessaires.

# Exercise 6 — Conversational Agent with Context Memory

## Selected technique: Structured History

Une mémoire structurée est adaptée à un coach virtuel parce qu’elle sépare :

- les préférences stables ;
- les objectifs ;
- les observations récentes ;
- les conseils déjà donnés ;
- les contraintes de sécurité.

Le modèle ne doit pas relire une conversation brute très longue à chaque
tour.

## 6.1 Structured memory schema

In [ ]:
@dataclass
class HealthCoachMemory:
    """Compact, user-approved memory for a virtual health coach."""

    user_id: str
    preferences: Dict[str, Any]
    goals: List[str]
    recent_observations: List[str]
    previous_advice: List[str]
    constraints: List[str]
    last_updated: str


health_memory = HealthCoachMemory(
    user_id="user_001",
    preferences={
        "exercise": "walking and short home workouts",
        "diet": "vegetarian",
        "communication": "short, encouraging messages",
    },
    goals=[
        "sleep at least 7 hours on most nights",
        "exercise 3 times per week",
    ],
    recent_observations=[
        "average sleep reported last week: 6 hours",
        "user completed two 25-minute walks",
        "late caffeine appears to affect sleep",
    ],
    previous_advice=[
        "avoid caffeine after 3:00 PM",
        "prepare a fixed 10:30 PM wind-down routine",
    ],
    constraints=[
        "no gym membership",
        "avoid medical diagnosis",
        "recommend professional care for alarming symptoms",
    ],
    last_updated="2026-07-15",
)

past_context = json.dumps(
    asdict(health_memory),
    indent=2,
)

print(past_context)

## 6.2 Prompt using the memory

In [ ]:
next_user_message = (
    "I only slept six hours again. What should I focus on tonight?"
)

next_turn_prompt = f"""
Act as a supportive virtual health coach.

Use the approved structured memory below to maintain continuity.

<memory>
{past_context}
</memory>

New user message:
<message>
{next_user_message}
</message>

Response requirements:
1. Acknowledge the new message in one sentence.
2. Refer to at most two relevant memory facts.
3. Give exactly three practical actions for tonight.
4. Do not repeat previous advice without adapting it to the new situation.
5. Respect the user's vegetarian preference, exercise preference, and lack
   of a gym membership when relevant.
6. Do not diagnose a condition or claim that a habit will cure a disorder.
7. If the message indicates severe symptoms, persistent major sleep
   disruption, breathing problems, chest pain, fainting, or immediate
   danger, recommend appropriate professional or urgent care.
8. End with one simple check-in question.
9. Keep the response under 140 words.

Memory policy:
- Treat memory as context, not as guaranteed medical fact.
- Do not mention hidden fields or the user ID.
- Do not infer sensitive information not stored in memory.
- Suggest a memory update only when the user provides a new stable
  preference, goal, or constraint.

Return only the user-facing response.
""".strip()

print(next_turn_prompt)

## 6.3 Memory update logic

La mémoire ne doit pas enregistrer automatiquement chaque phrase. Elle doit
conserver uniquement les informations durables, utiles et autorisées.

In [ ]:
def propose_memory_updates(
    current_memory: HealthCoachMemory,
    user_message: str,
) -> List[Dict[str, str]]:
    """Propose memory updates from explicit user statements.

    This demonstration uses simple rules. A production system would add:
    - user confirmation;
    - privacy controls;
    - expiration policies;
    - secure storage;
    - deletion and correction features.
    """
    normalized_message = user_message.lower()
    proposed_updates: List[Dict[str, str]] = []

    # Record only explicit, potentially stable preferences.
    if "i prefer morning workouts" in normalized_message:
        proposed_updates.append(
            {
                "field": "preferences.exercise_time",
                "value": "morning",
                "reason": "Explicit stable preference stated by user",
            }
        )

    if "my goal is" in normalized_message:
        proposed_updates.append(
            {
                "field": "goals",
                "value": user_message,
                "reason": "Explicit goal statement",
            }
        )

    # A one-night event should stay in recent observations, not preferences.
    if "last night" in normalized_message:
        proposed_updates.append(
            {
                "field": "recent_observations",
                "value": user_message,
                "reason": "Recent event, not assumed to be permanent",
            }
        )

    return proposed_updates


memory_update_example = propose_memory_updates(
    health_memory,
    "Last night I slept six hours, and I prefer morning workouts.",
)

print(json.dumps(memory_update_example, indent=2))

## 6.4 Why structured history was selected

Comparaison rapide :

| Technique | Avantage | Limite |
|---|---|---|
| Prior message passing | Simple | Devient long et coûteux |
| Structured history | Compact, contrôlable, explicable | Requiert une logique de mise à jour |
| Vector retrieval | Adapté aux longues conversations | Peut récupérer un souvenir hors contexte |

La mémoire structurée est la meilleure option ici parce que le nombre
d’informations utiles est limité et que la confidentialité exige un contrôle
précis de ce qui est conservé.

# Final comparison of prompt patterns

| Exercise | Pattern or technique | Main purpose |
|---|---|---|
| 1 | Verifiable calculation steps | Correct arithmetic reasoning |
| 2 | Few-shot classification | Stable label boundaries |
| 3 | Aligned multi-path reasoning | Cross-check independent answers |
| 4 | Prompt chaining | Multi-stage document processing |
| 5 | Role prompting + fairness rules | Reduce stereotypical recommendations |
| 6 | Structured memory | Personalized conversational continuity |

# Prompt engineering checklist

Avant de déployer un prompt :

1. La tâche est-elle précise ?
2. Les entrées sont-elles délimitées ?
3. Les sorties ont-elles un schéma clair ?
4. Les exemples couvrent-ils les cas ambigus ?
5. Les contraintes sont-elles vérifiables ?
6. Les erreurs ou informations manquantes ont-elles un comportement prévu ?
7. Une étape indépendante peut-elle vérifier le résultat ?
8. Les règles de sécurité et de biais sont-elles explicites ?
9. La mémoire conserve-t-elle uniquement les informations nécessaires ?
10. Une validation humaine est-elle prévue pour les situations sensibles ?

# Conclusion

Les six exercices montrent qu’un prompt avancé est souvent un petit système
de contrôle plutôt qu’une simple question.

Les éléments les plus importants sont :

- des instructions vérifiables ;
- des labels et formats fermés ;
- plusieurs chemins indépendants pour les calculs critiques ;
- une séparation claire des étapes ;
- des branches conditionnelles ;
- des règles explicites de fairness ;
- une mémoire minimale et structurée.

Un bon prompt ne garantit pas une sortie parfaite. Il rend toutefois le
comportement du modèle plus prévisible, testable et intégrable.